# Bulk Mode ID Workflow Demo

This notebook demonstrates how to:
1. Randomize a model
2. Analyze traps **and** eligible MP-bulk IDs
3. Select bulk IDs per layer
4. Remove bulk modes via `remove_modes(...)`
5. Remove trap IDs via `remove_traps(...)`


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import weightwatcher as ww


In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(64, 48, bias=False)
        self.fc2 = nn.Linear(48, 32, bias=False)
        with torch.no_grad():
            self.fc1.weight += 2.5 * torch.outer(torch.linspace(0.5, 1.5, 48), torch.linspace(-1.0, 1.0, 64))

    def forward(self, x):
        return self.fc2(self.fc1(x))

model = TinyNet()
watcher = ww.WeightWatcher(model=model)


In [ ]:
randomized_model, trap_state = watcher.randomize_model(model=model, rng=123, return_state=True)
df, trap_state = watcher.analyze_traps(
    randomized_model=randomized_model,
    trap_state=trap_state,
    return_artifacts=True,
    return_bulk_ids=True,
    max_bulk_modes_per_layer=10,
    bulk_sampling_seed=123,
    bulk_sampling_strategy='uniform',
    plot=False,
)
df.head()


In [ ]:
bulk_df = df.query("mode_type == 'bulk'").copy()
trap_df = df.query("mode_type == 'trap'").copy()
print('bulk rows:', len(bulk_df), 'trap rows:', len(trap_df))
bulk_df[['layer_id','bulk_id','svd_mode_index','eigenvalue']].head()


In [ ]:
bulk_ids_by_layer = (
    bulk_df.groupby('layer_id')['bulk_id']
    .apply(lambda s: list(map(int, s.head(2))))
    .to_dict()
)
bulk_ids_by_layer


In [ ]:
bulk_ablated_model = watcher.remove_modes(
    mode_ids_by_layer=bulk_ids_by_layer,
    mode_type='bulk',
    randomized_model=randomized_model,
    trap_state=trap_state,
    plot=False,
)
bulk_ablated_model


In [ ]:
if len(trap_df) > 0:
    lid = int(trap_df.iloc[0]['layer_id'])
    tid = int(trap_df.iloc[0]['trap_id'])
    trap_ablated_model = watcher.remove_traps(
        randomized_model=randomized_model,
        trap_state=trap_state,
        bulk_ids_by_layer=None,
        trap_indices=[tid],
        mode_type='trap',
        plot=False,
    )
    print('Removed trap_id', tid, 'from layer', lid)


## Notes
- Public `trap_id` and `bulk_id` are 1-based.
- Internal `svd_mode_index` is 0-based for debugging/mapping.
- `trap_state['layers'][layer_id]` stores per-layer mappings (`trap_id_to_svd_index`, `bulk_id_to_svd_index`).
